# Unified calculate_FCD

모든 모델(TransCVAE/LSTM_CVAE/Encoder_Only) 공통 FCD 계산 노트북입니다.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd
import torch
from rdkit import Chem, RDLogger
from fcd_torch import FCD

RDLogger.DisableLog('rdApp.error')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


In [ ]:
# ===== User Config =====
def _is_models_root(p: Path) -> bool:
    return (p / 'utils').is_dir() and (p / 'models').is_dir() and (p / 'notebooks').is_dir() and (p / 'data').is_dir()


def resolve_models_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd] + list(cwd.parents):
        if _is_models_root(base):
            return base
        cand = base / 'MY_PAPER_RELATED' / 'MODELS'
        if _is_models_root(cand):
            return cand
    raise FileNotFoundError('Could not locate MODELS root from current working directory.')


MODELS_ROOT = resolve_models_root()
CHECKPOINT_ROOT = MODELS_ROOT / 'checkpoints'

PRESETS = {
    'TransCVAE': {'variant': 'TransCVAE', 'family': 'cvae', 'backbone': 'trans', 'ckpt_tag': 'TransCVAE'},
    'TransCVAE_PSMILES': {'variant': 'TransCVAE_PSMILES', 'family': 'cvae', 'backbone': 'trans', 'ckpt_tag': 'TransCVAE_PSMILES'},
    'LSTM_CVAE': {'variant': 'LSTM_CVAE', 'family': 'cvae', 'backbone': 'lstm', 'ckpt_tag': 'LSTM_CVAE'},
    'LSTM_CVAE_PSMILES': {'variant': 'LSTM_CVAE_PSMILES', 'family': 'cvae', 'backbone': 'lstm', 'ckpt_tag': 'LSTM_CVAE_PSMILES'},
    'Encoder_Only': {'variant': 'Encoder_Only', 'family': 'encoder_only', 'decode_mode': 'Encoder_Only', 'ckpt_tag': 'Encoder_Only'},
    'Encoder_Only_PSMILES': {'variant': 'Encoder_Only_PSMILES', 'family': 'encoder_only', 'decode_mode': 'Encoder_Only', 'ckpt_tag': 'Encoder_Only_PSMILES'},
    'Encoder_Only_SELFIES_PSMILES': {'variant': 'Encoder_Only', 'family': 'encoder_only', 'decode_mode': 'Encoder_Only', 'ckpt_tag': 'Encoder_Only', 'output_tag': 'Encoder_Only_SELFIES_PSMILES'},
}

# Legacy aliases (for backward compatibility)
ALIASES = {
    'TransCVAE': 'TransCVAE',
    'TransCVAE_PSMILES': 'TransCVAE_PSMILES',
}

# ----------------------------
# Execution options
# ----------------------------
RUN_ALL_MODELS = False
MODEL_NAME = 'Encoder_Only_PSMILES'  # used when RUN_ALL_MODELS=False
MODEL_NAME_LIST = None       # used when RUN_ALL_MODELS=True; None => all PRESETS keys

RUN_ALL_CONDITIONS = True
cond_key = 'HIGH'            # used when RUN_ALL_CONDITIONS=False
COND_KEYS = ['LOW', 'HIGH']  # used when RUN_ALL_CONDITIONS=True

# FCD repeat evaluation options
FCD_REPEAT_EVALS = 5         # repeat full generation+FCD N times per model/condition
REPEAT_SEED_BASE = 2026

# Decoding policy options
# - "shared": same decoding params for all conditions
# - "condition_function": params determined by a predefined function of cond_value
DECODE_POLICY_MODE = 'shared'  # 'shared' | 'condition_function'

# Shared policy (used when DECODE_POLICY_MODE='shared')
DECODE_SHARED_TEMPERATURE = 1.0
DECODE_SHARED_TOP_K = None
DECODE_SHARED_TOP_P = 0.95

# Condition-function policy (used when DECODE_POLICY_MODE='condition_function')
# cond_value interpolation t in [0,1] is computed from COND_Z min/max.
DECODE_CF_TEMPERATURE_BASE = 0.95
DECODE_CF_TEMPERATURE_GAIN = 0.20
DECODE_CF_TOP_K_BASE = 60
DECODE_CF_TOP_K_GAIN = 40
DECODE_CF_TOP_P_BASE = 0.92
DECODE_CF_TOP_P_GAIN = 0.06

# Condition values used for generation input
# Scale: standardized log-space value from (log1p -> minmax -> z-score).
COND_Z = {'LOW': -0.54, 'HIGH': 12.0}

latent_dim = 128              # for cvae family
batch_size = 1024
num_rounds = 50               # rounds per repeat

# encoder_only settings
decoder_ckpt_variant = 'pi1m_finetuned_bundle'  # 'baseline' | 'pi1m_finetuned' | 'pi1m_finetuned_bundle' | 'custom'
decoder_custom_ckpt = None

output_root = MODELS_ROOT / 'FCD_runs'
output_root.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)


def resolve_encoder_only_ckpt_path(
    checkpoint_root: Path,
    mode_stem: str,
    ckpt_tag: str,
    ckpt_variant: str,
    custom_ckpt: str | None = None,
    fallback_tags=None,
) -> Path:
    suffix_map = {
        'baseline': '',
        'pi1m_finetuned': '_pi1m_finetuned',
        'pi1m_finetuned_bundle': '_pi1m_finetuned_bundle',
    }

    if ckpt_variant == 'custom':
        if custom_ckpt is None:
            raise ValueError('Set decoder_custom_ckpt when decoder_ckpt_variant=\"custom\"')
        p = Path(custom_ckpt)
        if not p.exists():
            raise FileNotFoundError(f'Custom checkpoint not found: {p}')
        return p

    if ckpt_variant not in suffix_map:
        raise ValueError(f'Unknown decoder_ckpt_variant: {ckpt_variant}')

    suffix = suffix_map[ckpt_variant]
    tags = [ckpt_tag] + list(fallback_tags or [])

    dedup_tags = []
    for t in tags:
        if t not in dedup_tags:
            dedup_tags.append(t)

    tried = []
    for t in dedup_tags:
        p = checkpoint_root / f'{mode_stem}_{t}{suffix}.pth'
        tried.append(str(p))
        if p.exists():
            if t != ckpt_tag:
                print(f'[CkptFallback] requested tag={ckpt_tag} -> using existing tag={t}')
            return p

    tried_msg = '\n - '.join([''] + tried)
    raise FileNotFoundError(f'Encoder_Only checkpoint not found. Tried:{tried_msg}')


def _resolve_model_keys():
    if RUN_ALL_MODELS:
        names = list(PRESETS.keys()) if MODEL_NAME_LIST is None else list(MODEL_NAME_LIST)
    else:
        names = [MODEL_NAME]

    keys = []
    for name in names:
        key = ALIASES.get(name, name)
        if key not in PRESETS:
            raise KeyError(f'Unknown model name: {name}. Available: {list(PRESETS.keys()) + list(ALIASES.keys())}')
        keys.append(key)
    return keys


def _resolve_cond_keys():
    keys = list(COND_KEYS) if RUN_ALL_CONDITIONS else [cond_key]
    for k in keys:
        if k not in COND_Z:
            raise KeyError(f'cond_key must be one of {list(COND_Z.keys())}, got: {k}')
    return keys


MODEL_KEYS_TO_RUN = _resolve_model_keys()
COND_KEYS_TO_RUN = _resolve_cond_keys()


def _clip01(x: float) -> float:
    return max(0.0, min(1.0, float(x)))


def resolve_decode_policy(cond_key_local: str, cond_value_local: float) -> dict:
    mode = str(DECODE_POLICY_MODE).strip().lower()
    if mode == 'shared':
        return {
            'policy_mode': 'shared',
            'temperature': float(DECODE_SHARED_TEMPERATURE),
            'top_k': None if DECODE_SHARED_TOP_K is None else int(DECODE_SHARED_TOP_K),
            'top_p': None if DECODE_SHARED_TOP_P is None else float(DECODE_SHARED_TOP_P),
            'cond_interp': None,
        }

    if mode == 'condition_function':
        cond_vals = [float(v) for v in COND_Z.values()]
        lo = min(cond_vals)
        hi = max(cond_vals)
        if hi > lo:
            t = _clip01((float(cond_value_local) - lo) / (hi - lo))
        else:
            t = 0.5

        temperature = float(DECODE_CF_TEMPERATURE_BASE) + float(DECODE_CF_TEMPERATURE_GAIN) * t
        top_k = int(round(float(DECODE_CF_TOP_K_BASE) + float(DECODE_CF_TOP_K_GAIN) * t))
        top_p = float(DECODE_CF_TOP_P_BASE) + float(DECODE_CF_TOP_P_GAIN) * t
        top_p = _clip01(top_p)
        if top_p <= 0.0:
            top_p = None

        return {
            'policy_mode': 'condition_function',
            'temperature': max(1e-4, float(temperature)),
            'top_k': max(1, int(top_k)),
            'top_p': top_p,
            'cond_interp': float(t),
        }

    raise ValueError(f"Unknown DECODE_POLICY_MODE={DECODE_POLICY_MODE}. Use 'shared' or 'condition_function'.")


# Backward-compatible single-model variables (used by earlier cells)
model_key = ALIASES.get(MODEL_NAME, MODEL_NAME)
if model_key not in PRESETS:
    raise KeyError(f'MODEL_NAME must be one of: {list(PRESETS.keys()) + list(ALIASES.keys())}, got: {MODEL_NAME}')
cfg = PRESETS[model_key].copy()
MODEL_VARIANT = cfg['variant']
MODEL_ROOT = MODELS_ROOT

if not MODEL_ROOT.exists():
    raise FileNotFoundError(f'MODEL_ROOT not found: {MODEL_ROOT}')

print('MODELS_ROOT:', MODELS_ROOT)
print('CHECKPOINT_ROOT:', CHECKPOINT_ROOT)
print('MODEL_ROOT:', MODEL_ROOT)
print('MODEL_KEYS_TO_RUN:', MODEL_KEYS_TO_RUN)
print('COND_KEYS_TO_RUN:', COND_KEYS_TO_RUN)
print('FCD_REPEAT_EVALS:', FCD_REPEAT_EVALS)
print('COND_Z:', COND_Z)
print('DECODE_POLICY_MODE:', DECODE_POLICY_MODE)




In [ ]:
import os
# ===== Project Local Imports =====
def purge_project_modules():
    to_remove = []
    for name in list(sys.modules.keys()):
        if name == 'utils' or name.startswith('utils.') or name == 'models' or name.startswith('models.'):
            to_remove.append(name)
    for name in to_remove:
        del sys.modules[name]

purge_project_modules()
os.environ['MODELS_VARIANT'] = MODEL_VARIANT

if str(MODELS_ROOT) in sys.path:
    sys.path.remove(str(MODELS_ROOT))
sys.path.insert(0, str(MODELS_ROOT))

from utils.utils import *
from utils.dataloader import dataset
from utils.generate import generate_batch_sequence

vocab = dataset.vocab
index_to_token = {idx: token for token, idx in vocab.items()}

print('dataset size:', len(dataset))
print('vocab size:', dataset.vocab_size)
print('max_len:', dataset.max_len)
print('MODELS_VARIANT:', os.environ.get('MODELS_VARIANT'))


In [ ]:
# ===== Model Loader =====
cond_value = float(COND_Z[cond_key])

prop_mode = str(getattr(dataset, 'property_mode', '')).strip().lower()
if cfg['family'] == 'encoder_only' and prop_mode != 'log_minmax_zscore':
    raise ValueError(
        f"Encoder_Only requires property_mode='log_minmax_zscore' for COND_Z in standardized log space. Got: {prop_mode}"
    )

def make_condition(batch_n: int):
    return torch.full((batch_n, 1, 1), cond_value, dtype=torch.float32, device=device)

if cfg['family'] == 'encoder_only':
    from models.Trans import Encoder_Only

    mode = cfg.get('decode_mode', 'Encoder_Only')
    model_cls = Encoder_Only
    model = model_cls(vocab_size=dataset.vocab_size).to(device).eval()

    ckpt_tag = cfg.get('ckpt_tag', MODEL_NAME)
    mode_stem = 'encoder_only' if mode == 'Encoder_Only' else 'encoder_only'
    fallback_tags = ['Encoder_Only'] if ('SELFIES' in MODEL_NAME or 'SELFIES' in ckpt_tag) else []
    ckpt_path = resolve_encoder_only_ckpt_path(
        checkpoint_root=CHECKPOINT_ROOT,
        mode_stem=mode_stem,
        ckpt_tag=ckpt_tag,
        ckpt_variant=decoder_ckpt_variant,
        custom_ckpt=decoder_custom_ckpt,
        fallback_tags=fallback_tags,
    )

    blob = torch.load(ckpt_path, map_location=device)
    _ = load_encoder_only_checkpoint_compat(model, blob, current_vocab=dataset.vocab)

    forbidden_token_ids = []

    def sample_generation_input(batch_n: int):
        return make_condition(batch_n)

    print('Loaded decoder-only checkpoint:', ckpt_path)

elif cfg['family'] == 'cvae':
    ckpt_tag = cfg.get('ckpt_tag', MODEL_NAME)
    if cfg['backbone'] == 'trans':
        from models.Trans import CVAE, PriorNet
        model = CVAE(latent_dim=latent_dim).to(device).eval()
        prior = PriorNet(y_dim=1, latent_dim=latent_dim).to(device).eval()
        model_path = CHECKPOINT_ROOT / f'model_weights_trans_{ckpt_tag}.pth'
        prior_path = CHECKPOINT_ROOT / f'model_weights_prior_trans_{ckpt_tag}.pth'
    elif cfg['backbone'] == 'lstm':
        from models.LSTM import LSTMCVAE, LSTMPriorNet
        model = LSTMCVAE(latent_dim=latent_dim).to(device).eval()
        prior = LSTMPriorNet(y_dim=1, latent_dim=latent_dim).to(device).eval()
        model_path = CHECKPOINT_ROOT / f'model_weights_lstm_{ckpt_tag}.pth'
        prior_path = CHECKPOINT_ROOT / f'model_weights_prior_lstm_{ckpt_tag}.pth'
    else:
        raise ValueError(f"Unknown backbone: {cfg['backbone']}")

    model.load_state_dict(torch.load(model_path, map_location=device), strict=False)
    prior.load_state_dict(torch.load(prior_path, map_location=device), strict=False)

    forbidden_token_ids = None

    def sample_generation_input(batch_n: int):
        cond = make_condition(batch_n)
        mean, var = prior(cond.squeeze(-1))
        z = model.reparameterize(mean, var)
        return z

    print('Loaded cvae model:', model_path)
    print('Loaded cvae prior:', prior_path)
else:
    raise ValueError(f"Unknown family: {cfg['family']}")


In [ ]:
# ===== FCD Helpers =====
def canonicalize_smiles(smiles):
    try:
        sm = PS(smiles).canonicalize.psmiles
        if Chem.MolFromSmiles(sm) is None:
            return None
        return sm
    except Exception:
        return None


def is_valid(smiles):
    return Chem.MolFromSmiles(smiles) is not None


# utils.utils에 없는 경우를 대비한 fallback
if 'tok_ids_to_smiles' not in globals():
    def tok_ids_to_smiles(tok_ids, id2tok):
        tokens = [id2tok.get(int(i), '') for i in tok_ids]
        if '[EOS]' in tokens:
            tokens = tokens[:tokens.index('[EOS]')]
        tokens = [t for t in tokens if t not in {'[SOS]', '[PAD]', '[EOS]'}]
        if not tokens:
            return None

        raw = ''.join(tokens)
        sm = canonicalize_smiles(raw)
        if sm is not None:
            return sm
        # canonicalize 실패 시 raw가 유효하면 raw 반환
        return raw if is_valid(raw) else None

if 'filter_valid_unique' not in globals():
    def filter_valid_unique(smiles_list):
        valid_unique = []
        seen = set()
        invalid_or_empty = 0

        for sm in smiles_list:
            sm = (str(sm).strip() if sm is not None else '')
            if not sm:
                invalid_or_empty += 1
                continue

            can = canonicalize_smiles(sm)
            if can is None:
                invalid_or_empty += 1
                continue

            if can not in seen:
                seen.add(can)
                valid_unique.append(can)

        return valid_unique, invalid_or_empty


def tokens_to_smiles_list(generated_tokens, index_to_token):
    generated_smiles = []
    for row in generated_tokens:
        row = list(row)
        if row and row[0] == vocab['[SOS]']:
            row = row[1:]
        if row and row[-1] == vocab['[EOS]']:
            row = row[:-1]

        smiles = tok_ids_to_smiles(row, index_to_token)
        generated_smiles.append(smiles or '')

    generated_smiles = [sm.strip() for sm in generated_smiles]
    return generated_smiles

source_csv = MODELS_ROOT / 'data' / 'simulation-trajectory-aggregate_aligned.csv'
original_smiles = pd.read_csv(source_csv).iloc[:, 1].astype(str).tolist()
original_smiles_valid, _ = filter_valid_unique(original_smiles)
original_smiles_set = set(original_smiles_valid)

calc = FCD(device=str(device), n_jobs=8)
print('original valid:', len(original_smiles_valid))


In [ ]:
# ===== Run Generation + FCD =====
import importlib
import random


def _slug(text):
    s = str(text).strip().lower()
    for old, new in [(' ', '_'), ('[', ''), (']', ''), ('/', '_'), ('.', 'p'), ('-', 'm'), ('+', 'p')]:
        s = s.replace(old, new)
    return s


def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_runtime(model_key_local: str, cond_key_local: str):
    cfg_local = PRESETS[model_key_local].copy()
    model_variant_local = cfg_local['variant']
    output_tag_local = cfg_local.get('output_tag', model_key_local)
    cond_value_local = float(COND_Z[cond_key_local])
    decode_cfg_local = resolve_decode_policy(cond_key_local, cond_value_local)

    # Refresh project modules per model variant
    purge_project_modules()
    os.environ['MODELS_VARIANT'] = model_variant_local
    if str(MODELS_ROOT) in sys.path:
        sys.path.remove(str(MODELS_ROOT))
    sys.path.insert(0, str(MODELS_ROOT))

    u = importlib.import_module('utils.utils')
    dloader = importlib.import_module('utils.dataloader')
    gmod = importlib.import_module('utils.generate')

    dataset_local = dloader.dataset
    generate_batch_sequence_local = gmod.generate_batch_sequence
    vocab_local = dataset_local.vocab
    index_to_token_local = {idx: token for token, idx in vocab_local.items()}

    device_local = u.device

    # COND_Z values are assumed to be in standardized log space
    # (log1p -> minmax -> z-score), matching property_mode='log_minmax_zscore'.
    prop_mode_local = str(getattr(dataset_local, 'property_mode', '')).strip().lower()
    if cfg_local['family'] == 'encoder_only' and prop_mode_local != 'log_minmax_zscore':
        raise ValueError(
            f"Encoder_Only requires property_mode='log_minmax_zscore' for COND_Z. Got: {prop_mode_local}"
        )

    def make_condition(batch_n: int):
        return torch.full((batch_n, 1, 1), cond_value_local, dtype=torch.float32, device=device_local)

    if cfg_local['family'] == 'encoder_only':
        mtrans = importlib.import_module('models.Trans')
        model_cls = mtrans.Encoder_Only
        model_local = model_cls(vocab_size=dataset_local.vocab_size).to(device_local).eval()

        ckpt_tag = cfg_local.get('ckpt_tag', model_key_local)
        mode = cfg_local.get('decode_mode', 'Encoder_Only')
        mode_stem = 'encoder_only' if mode == 'Encoder_Only' else 'encoder_only'
        fallback_tags = ['Encoder_Only'] if ('SELFIES' in model_key_local or 'SELFIES' in ckpt_tag) else []
        ckpt_path = resolve_encoder_only_ckpt_path(
            checkpoint_root=CHECKPOINT_ROOT,
            mode_stem=mode_stem,
            ckpt_tag=ckpt_tag,
            ckpt_variant=decoder_ckpt_variant,
            custom_ckpt=decoder_custom_ckpt,
            fallback_tags=fallback_tags,
        )

        blob = torch.load(ckpt_path, map_location=device_local)
        _ = u.load_encoder_only_checkpoint_compat(model_local, blob, current_vocab=dataset_local.vocab)
        forbidden_token_ids_local = []

        def sample_generation_input_local(batch_n: int):
            return make_condition(batch_n)

        print(f"[Loader] {model_key_local} loaded: {ckpt_path}")

    elif cfg_local['family'] == 'cvae':
        ckpt_tag = cfg_local.get('ckpt_tag', model_key_local)
        if cfg_local['backbone'] == 'trans':
            mtrans = importlib.import_module('models.Trans')
            model_local = mtrans.CVAE(latent_dim=latent_dim).to(device_local).eval()
            prior_local = mtrans.PriorNet(y_dim=1, latent_dim=latent_dim).to(device_local).eval()
            model_path = CHECKPOINT_ROOT / f'model_weights_trans_{ckpt_tag}.pth'
            prior_path = CHECKPOINT_ROOT / f'model_weights_prior_trans_{ckpt_tag}.pth'
        elif cfg_local['backbone'] == 'lstm':
            mlstm = importlib.import_module('models.LSTM')
            model_local = mlstm.LSTMCVAE(latent_dim=latent_dim).to(device_local).eval()
            prior_local = mlstm.LSTMPriorNet(y_dim=1, latent_dim=latent_dim).to(device_local).eval()
            model_path = CHECKPOINT_ROOT / f'model_weights_lstm_{ckpt_tag}.pth'
            prior_path = CHECKPOINT_ROOT / f'model_weights_prior_lstm_{ckpt_tag}.pth'
        else:
            raise ValueError(f"Unknown backbone: {cfg_local['backbone']}")

        model_local.load_state_dict(torch.load(model_path, map_location=device_local), strict=False)
        prior_local.load_state_dict(torch.load(prior_path, map_location=device_local), strict=False)
        forbidden_token_ids_local = None

        def sample_generation_input_local(batch_n: int):
            cond = make_condition(batch_n)
            mean, var = prior_local(cond.squeeze(-1))
            z = model_local.reparameterize(mean, var)
            return z

        print(f"[Loader] {model_key_local} loaded: {model_path} / {prior_path}")
    else:
        raise ValueError(f"Unknown family: {cfg_local['family']}")

    # local token->smiles helper (uses local vocab)
    def tokens_to_smiles_list_local(generated_tokens):
        generated_smiles_local = []
        for row in generated_tokens:
            row = list(row)
            if row and row[0] == vocab_local['[SOS]']:
                row = row[1:]
            if row and row[-1] == vocab_local['[EOS]']:
                row = row[:-1]
            sm = tok_ids_to_smiles(row, index_to_token_local)
            generated_smiles_local.append(sm or '')
        generated_smiles_local = [sm.strip() for sm in generated_smiles_local]
        return generated_smiles_local

    print(f"[Decode] mode={decode_cfg_local['policy_mode']} cond={cond_key_local} effective_cond_value={cond_value_local} temp={decode_cfg_local['temperature']:.4f} top_k={decode_cfg_local['top_k']} top_p={decode_cfg_local['top_p']} t={decode_cfg_local['cond_interp']}")

    return {
        'cfg': cfg_local,
        'model_variant': model_variant_local,
        'output_tag': output_tag_local,
        'cond_key': cond_key_local,
        'cond_value': cond_value_local,
        'decode_cfg': decode_cfg_local,
        'device': device_local,
        'dataset': dataset_local,
        'vocab': vocab_local,
        'model': model_local,
        'forbidden_token_ids': forbidden_token_ids_local,
        'sample_generation_input': sample_generation_input_local,
        'generate_batch_sequence': generate_batch_sequence_local,
        'tokens_to_smiles_list': tokens_to_smiles_list_local,
    }


def run_one_repeat(runtime: dict, repeat_idx: int, cond_criterion_label: str, output_dir: Path):
    model_local = runtime['model']
    dataset_local = runtime['dataset']
    vocab_local = runtime['vocab']
    sample_generation_input_local = runtime['sample_generation_input']
    generate_batch_sequence_local = runtime['generate_batch_sequence']
    forbidden_token_ids_local = runtime['forbidden_token_ids']
    tokens_to_smiles_list_local = runtime['tokens_to_smiles_list']
    device_local = runtime['device']
    decode_cfg_local = runtime['decode_cfg']

    repeat_dir = output_dir / f'repeat_{repeat_idx:02d}'
    repeat_dir.mkdir(parents=True, exist_ok=True)

    calc_local = FCD(device=str(device_local), n_jobs=8)

    run_rows = []
    all_novel_set = set()
    all_novel_list = []

    for run_idx in range(1, num_rounds + 1):
        model_input = sample_generation_input_local(batch_size)

        gen_kwargs = dict(
            max_length=dataset_local.max_len + 2,
            start_token=vocab_local['[SOS]'],
            end_token=vocab_local['[EOS]'],
            pad_token=vocab_local['[PAD]'],
            temperature=float(decode_cfg_local['temperature']),
            top_k=decode_cfg_local['top_k'],
            top_p=decode_cfg_local['top_p'],
            device=str(device_local),
        )
        if forbidden_token_ids_local is not None:
            gen_kwargs['forbidden_token_ids'] = forbidden_token_ids_local

        generated_tokens = generate_batch_sequence_local(model_local, model_input, **gen_kwargs)
        generated_smiles = tokens_to_smiles_list_local(generated_tokens)

        # valid: duplicate-preserving count of valid molecules
        valid_smiles = []
        for sm in generated_smiles:
            sm = (str(sm).strip() if sm is not None else '')
            if not sm:
                continue
            can = canonicalize_smiles(sm)
            if can is not None:
                valid_smiles.append(can)
        valid_generated_count = len(valid_smiles)

        # valid_unique: unique valid molecules
        valid_generated_smiles, invalid_or_empty = filter_valid_unique(generated_smiles)
        # valid_unique_novel: unique valid molecules that are novel vs reference set
        novel_generated_smiles = [sm for sm in valid_generated_smiles if sm not in original_smiles_set]

        for sm in novel_generated_smiles:
            if sm not in all_novel_set:
                all_novel_set.add(sm)
                all_novel_list.append(sm)

        run_path = repeat_dir / f'run_{run_idx:02d}_novel_smiles_{cond_criterion_label}.csv'
        pd.DataFrame({'smiles': novel_generated_smiles}).to_csv(run_path, index=False)

        run_rows.append({
            'repeat': repeat_idx,
            'run': run_idx,
            'batch_size': batch_size,
            'decode_policy_mode': decode_cfg_local['policy_mode'],
            'decode_temperature': float(decode_cfg_local['temperature']),
            'decode_top_k': decode_cfg_local['top_k'],
            'decode_top_p': decode_cfg_local['top_p'],
            'valid': valid_generated_count,
            'valid_unique': len(valid_generated_smiles),
            'valid_unique_novel': len(novel_generated_smiles),
            'invalid_or_empty': invalid_or_empty,
            'output_path': str(run_path),
        })

        print(f"[Repeat {repeat_idx:02d} | Run {run_idx:02d}] valid={valid_generated_count} valid_unique={len(valid_generated_smiles)} valid_unique_novel={len(novel_generated_smiles)} invalid_or_empty={invalid_or_empty}")

    all_novel_path = repeat_dir / f'all_novel_smiles_{cond_criterion_label}.csv'
    pd.DataFrame({'smiles': all_novel_list}).to_csv(all_novel_path, index=False)

    if len(all_novel_list) > 0:
        fcd_all = calc_local(all_novel_list, original_smiles_valid)
    else:
        fcd_all = float('nan')

    run_df = pd.DataFrame(run_rows)
    run_df['all_novel_unique'] = len(all_novel_list)
    run_df['fcd_all_novel_vs_original'] = fcd_all

    run_summary_path = repeat_dir / f'fcd_summary_{cond_criterion_label}_repeat_{repeat_idx:02d}.csv'
    run_df.to_csv(run_summary_path, index=False)

    repeat_summary = {
        'decode_policy_mode': decode_cfg_local['policy_mode'],
        'decode_temperature': float(decode_cfg_local['temperature']),
        'decode_top_k': decode_cfg_local['top_k'],
        'decode_top_p': decode_cfg_local['top_p'],
        'repeat': repeat_idx,
        'all_novel_unique': len(all_novel_list),
        'fcd_all_novel_vs_original': fcd_all,
        'all_novel_path': str(all_novel_path),
        'run_summary_path': str(run_summary_path),
    }
    return repeat_summary


all_final_rows = []

for model_key_local in MODEL_KEYS_TO_RUN:
    for cond_key_local in COND_KEYS_TO_RUN:
        print("=" * 90)
        print(f"[Start] model={model_key_local} cond={cond_key_local}")

        runtime = build_runtime(model_key_local, cond_key_local)
        model_variant_local = runtime['model_variant']
        output_tag_local = runtime.get('output_tag', model_key_local)

        cond_criterion_label = f"condz_{_slug(cond_key_local)}"
        output_dir = output_root / output_tag_local / cond_criterion_label
        output_dir.mkdir(parents=True, exist_ok=True)

        repeat_rows = []
        for rep_idx in range(1, int(FCD_REPEAT_EVALS) + 1):
            set_all_seeds(int(REPEAT_SEED_BASE) + rep_idx)
            rep_summary = run_one_repeat(runtime, rep_idx, cond_criterion_label, output_dir)
            rep_summary['model_name'] = model_variant_local
            rep_summary['model_key'] = model_key_local
            rep_summary['cond_criterion'] = cond_criterion_label
            rep_summary['cond_key'] = cond_key_local
            rep_summary['num_rounds'] = num_rounds
            rep_summary['batch_size'] = batch_size
            rep_summary['original_valid_unique'] = len(original_smiles_valid)
            repeat_rows.append(rep_summary)

            print(f"[Repeat {rep_idx:02d}] all_novel_unique={rep_summary['all_novel_unique']} fcd_all={rep_summary['fcd_all_novel_vs_original']}")

        repeats_df = pd.DataFrame(repeat_rows)
        repeats_path = output_dir / f'fcd_repeats_{cond_criterion_label}.csv'
        repeats_df.to_csv(repeats_path, index=False)

        fcd_series = pd.to_numeric(repeats_df['fcd_all_novel_vs_original'], errors='coerce')
        all_novel_series = pd.to_numeric(repeats_df['all_novel_unique'], errors='coerce')

        final_row = {
            'decode_policy_mode': repeats_df['decode_policy_mode'].iloc[0] if len(repeats_df) > 0 else None,
            'decode_temperature': float(repeats_df['decode_temperature'].iloc[0]) if len(repeats_df) > 0 else float('nan'),
            'decode_top_k': repeats_df['decode_top_k'].iloc[0] if len(repeats_df) > 0 else None,
            'decode_top_p': repeats_df['decode_top_p'].iloc[0] if len(repeats_df) > 0 else None,
            'model_name': output_tag_local,
            'model_key': model_key_local,
            'cond_criterion': cond_criterion_label,
            'cond_key': cond_key_local,
            'repeat_evals': int(FCD_REPEAT_EVALS),
            'batch_size': batch_size,
            'num_rounds': num_rounds,
            'original_valid_unique': len(original_smiles_valid),
            'fcd_all_mean': float(fcd_series.mean()),
            'fcd_all_std': float(fcd_series.std(ddof=1)) if len(fcd_series.dropna()) > 1 else float('nan'),
            'all_novel_unique_mean': float(all_novel_series.mean()),
            'all_novel_unique_std': float(all_novel_series.std(ddof=1)) if len(all_novel_series.dropna()) > 1 else float('nan'),
            'repeats_path': str(repeats_path),
        }

        final_summary_df = pd.DataFrame([final_row])
        final_summary_path = output_dir / f'final_summary_{cond_criterion_label}.csv'
        final_summary_df.to_csv(final_summary_path, index=False)

        print(f"[Done] model={model_key_local} cond={cond_key_local}")
        print(f"FCD(all novel vs original): mean={final_row['fcd_all_mean']:.6f}, std={final_row['fcd_all_std']:.6f}")
        print('saved repeats:', repeats_path)
        print('saved final  :', final_summary_path)

        all_final_rows.append(final_row)

all_final_df = pd.DataFrame(all_final_rows)
master_path = output_root / 'final_summary_all_models_repeated.csv'
all_final_df.to_csv(master_path, index=False)
print('=' * 90)
print('Saved master summary:', master_path)
all_final_df